In [ ]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

# Quant data policy: this repo reads Google Drive data; downloads run from note.
import sys
from pathlib import Path
_QUANT_ROOT = QUANT_ROOT
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
import cloud_data


In [ ]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

%load_ext autoreload
%autoreload 2

import gc
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(QUANT_ROOT))

from utils import (
    compact_daily_price_parquet,
    infer_stock_ids_from_kbar_dir,
    load_no_price_pairs,
    load_or_build_above_ma_signal,
    read_kbar_ids,
    read_kbar_with_supplement,
    build_kbar_coverage_report,
    summarize_kbar_coverage,
    signal_row_to_stock_ids,
)
from cloud_data import (
    DATA_ROOT,
    TRADING_ROOT,
    TW_STOCK_DAILY_PRICE,
    TW_STOCK_KBAR_1MIN,
    TW_STOCK_KBAR_ABOVE_MA60,
    TW_STOCK_SECTOR_MAP,
    TW_FUTURES_TX,
)


## get data

### config

In [3]:
start_date = "2022-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

ma_window = 60
retry_wait = 60
max_retries = 1
rate_limit_wait = 3660
rate_limit_max_retries = 2
api_batch_size = 30
rebuild_signal = True

# Fetch extra daily history so the first target date can have a valid MA60.
daily_start_date = (
    pd.Timestamp(start_date) - pd.Timedelta(days=ma_window * 3)
).strftime("%Y-%m-%d")

data_dir = TRADING_ROOT
stock_universe_output = DATA_ROOT / "trading" / "market_reference" / "note_data" / "stock_universe.parquet"
daily_output = TW_STOCK_DAILY_PRICE
full_kbar_dir = TW_STOCK_KBAR_1MIN
signal_output = full_kbar_dir / "above_ma60_signal.parquet"
kbar_output_dir = TW_STOCK_KBAR_ABOVE_MA60
missing_kbar_output = kbar_output_dir / "missing_kbar_ids.json"
missing_full_kbar_output = kbar_output_dir / "missing_full_kbar_dates.json"
kbar_output_dir.mkdir(parents=True, exist_ok=True)

### get daily data

In [ ]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

# 1. Read source data prepared by note.
if not daily_output.exists():
    raise FileNotFoundError(
        f"{daily_output} is missing. Run the daily data updater from {NOTE_REPO_ROOT}."
    )

daily = pd.read_parquet(daily_output)
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.loc[
    (daily["date"] >= daily_start_date)
    & (daily["date"] <= end_date)
].copy()
stock_ids = sorted(daily["stock_id"].astype(str).unique())

print(f"Valid stock count: {len(stock_ids):,}")
print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")


In [17]:
# Optional: re-compact the current daily_stock_price.parquet without calling API.
old_daily_output = daily_output

if not old_daily_output.exists():
    raise FileNotFoundError(old_daily_output)

if stock_universe_output.exists():
    stock_ids_for_compact = pd.read_parquet(stock_universe_output)["stock_id"].astype(str)
    stock_ids_for_compact = stock_ids_for_compact[stock_ids_for_compact.str.fullmatch(r"[1-9]\d{3}")].tolist()
else:
    stock_ids_for_compact = infer_stock_ids_from_kbar_dir(full_kbar_dir)
    if not stock_ids_for_compact:
        raise FileNotFoundError(
            "No stock_universe.parquet and no kbar/1min parquet files to infer stock ids from."
        )
    pd.DataFrame({"stock_id": stock_ids_for_compact}).to_parquet(
        stock_universe_output,
        index=False,
    )

print(f"Compacting {len(stock_ids_for_compact):,} stock ids")

daily = compact_daily_price_parquet(
    source_file=old_daily_output,
    output_file=daily_output,
    stock_ids=stock_ids_for_compact,
)

print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

Compacting 2,138 stock ids


Compact daily price: 100%|██████████| 5/5 [00:00<00:00, 10.62batch/s]


Daily rows: 2,272,554
Daily dates: 1,191
Daily stocks: 2,022
Daily range: 2021-07-05 ~ 2026-05-29


### build 60ma signal

In [7]:
# 2. Build/load date x stock_id signal matrix.
# True means this stock passed yesterday's close > MA60 condition for today's kbar.
if "daily" not in globals():
    daily = pd.read_parquet(
        daily_output,
        columns=["date", "stock_id", "close"],
    )

signal = load_or_build_above_ma_signal(
    daily=daily,
    output_file=signal_output,
    start_date=start_date,
    end_date=end_date,
    ma_window=ma_window,
    rebuild=rebuild_signal,
)

print(f"Signal shape: {signal.shape[0]} dates x {signal.shape[1]} stocks")
print(f"Signal true cells: {int(signal.to_numpy().sum())}")

del daily
gc.collect()

Signal shape: 1094 dates x 2052 stocks
Signal true cells: 949415


0

### get minute kbar data

In [ ]:
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

coverage = build_kbar_coverage_report(
    signal=signal,
    full_kbar_dir=full_kbar_dir,
    supplement_dir=kbar_output_dir,
)
missing = coverage.loc[coverage["missing_count"] > 0].copy()
display(missing.head(30))
print(
    "Quant does not download missing k-bars. Update them from note with: "
    "python scripts/data_updates/fetch_tw_stock_kbar_1min.py"
)


In [ ]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

print(
    "K-bar refresh moved to {NOTE_REPO_ROOT}/scripts/data_updates/"
    "fetch_tw_stock_kbar_1min.py"
)


In [11]:
# Optional inspection
# signal = pd.read_parquet(TW_STOCK_KBAR_1MIN / "2025-03-05.parquet")
signal = pd.read_parquet(TW_STOCK_SECTOR_MAP)
signal

,stock_id,sector
0,3687,文化創意業
1,3629,光電業
2,5481,電子零組件業
3,5450,電腦及週邊設備業
4,6238,其他電子類
...,...,...
2669,3024,光電業
2670,3025,通信網路業
2671,3026,電子工業
2672,3027,通信網路業


### inspection

In [ ]:
# Check whether the MA60 signal universe is covered by full kbar + supplement files.
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

coverage = build_kbar_coverage_report(
    signal=signal,
    full_kbar_dir=full_kbar_dir,
    supplement_dir=kbar_output_dir,
)
missing_full_top, incomplete_full_top = summarize_kbar_coverage(coverage, top_n=30)

missing_full_count = int((~coverage["full_exists"]).sum())
incomplete_full_count = int((coverage["full_exists"] & (coverage["missing_count"] > 0)).sum())

print(f"Coverage dates: {len(coverage):,}")
print(f"Missing full kbar dates: {missing_full_count:,}")
print(f"Full exists but signal still missing kbar: {incomplete_full_count:,}")

print("\nTop missing full kbar dates:")
display(missing_full_top)

print("\nTop incomplete full kbar dates:")
display(incomplete_full_top)

# Inspect one date by setting inspect_date, for example: inspect_date = "2025-02-14"
inspect_date = None
if inspect_date is not None:
    inspect_row = coverage.loc[coverage["date"] == inspect_date].iloc[0]
    inspect_missing_ids = inspect_row["missing_ids"]
    print(f"{inspect_date} missing ids: {len(inspect_missing_ids):,}")
    print(inspect_missing_ids[:100])

Coverage dates: 1,094
Missing full kbar dates: 942
Full exists but signal still missing kbar: 120

Top missing full kbar dates:


,date,signal_count,full_exists,full_count,supplement_count,covered_count,missing_count
1093,2026-07-13,941,False,0,941,941,0
1092,2026-07-09,983,False,0,972,972,11
1091,2026-07-08,1015,False,0,1004,1004,11
1090,2026-07-07,1204,False,0,1204,1204,0
1089,2026-07-06,1193,False,0,1193,1193,0
1088,2026-07-03,1052,False,0,1052,1052,0
1087,2026-07-02,968,False,0,968,968,0
1086,2026-07-01,978,False,0,978,978,0
1085,2026-06-30,898,False,0,898,898,0
1084,2026-06-29,829,False,0,829,829,0



Top incomplete full kbar dates:


,date,signal_count,full_exists,full_count,supplement_count,covered_count,missing_count
764,2025-03-06,1246,True,2179,0,1194,52
763,2025-03-05,1165,True,2177,0,1116,49
594,2024-06-19,1267,True,2128,0,1219,48
836,2025-06-20,762,True,2210,0,715,47
751,2025-02-14,1040,True,106,937,993,47
885,2025-08-28,1017,True,2229,0,970,47
691,2024-11-12,762,True,2144,0,715,47
860,2025-07-24,752,True,2216,0,706,46
861,2025-07-25,752,True,2214,0,706,46
884,2025-08-27,971,True,2240,0,925,46


## backtest

### resample

In [ ]:
from resample_kbar import resample_kbar_to_1h
from cloud_data import TW_STOCK_KBAR_1MIN, TW_STOCK_KBAR_ABOVE_MA60, TW_STOCK_KBAR_1H

resample_kbar_to_1h(
    source_dir=TW_STOCK_KBAR_1MIN,
    output_dir=TW_STOCK_KBAR_1H,
    supplement_dir=TW_STOCK_KBAR_ABOVE_MA60,
    start_date="2022-01-01",
    end_date=None
)

待處理: 1094 天（已有 0 天）


Resample 1min→1H: 100%|██████████| 1094/1094 [40:51<00:00,  2.24s/day]

完成: 1094 天，錯誤: 0 天


['2022-01-03',
 '2022-01-04',
 '2022-01-05',
 '2022-01-06',
 '2022-01-07',
 '2022-01-10',
 '2022-01-11',
 '2022-01-12',
 '2022-01-13',
 '2022-01-14',
 '2022-01-17',
 '2022-01-18',
 '2022-01-19',
 '2022-01-20',
 '2022-01-21',
 '2022-01-24',
 '2022-01-25',
 '2022-01-26',
 '2022-02-07',
 '2022-02-08',
 '2022-02-09',
 '2022-02-10',
 '2022-02-11',
 '2022-02-14',
 '2022-02-15',
 '2022-02-16',
 '2022-02-17',
 '2022-02-18',
 '2022-02-21',
 '2022-02-22',
 '2022-02-23',
 '2022-02-24',
 '2022-02-25',
 '2022-03-01',
 '2022-03-02',
 '2022-03-03',
 '2022-03-04',
 '2022-03-07',
 '2022-03-08',
 '2022-03-09',
 '2022-03-10',
 '2022-03-11',
 '2022-03-14',
 '2022-03-15',
 '2022-03-16',
 '2022-03-17',
 '2022-03-18',
 '2022-03-21',
 '2022-03-22',
 '2022-03-23',
 '2022-03-24',
 '2022-03-25',
 '2022-03-28',
 '2022-03-29',
 '2022-03-30',
 '2022-03-31',
 '2022-04-01',
 '2022-04-06',
 '2022-04-07',
 '2022-04-08',
 '2022-04-11',
 '2022-04-12',
 '2022-04-13',
 '2022-04-14',
 '2022-04-15',
 '2022-04-18',
 '2022-04-

#### inspection

In [ ]:
# 檢查是不是所有股票都被 resample

from resample_kbar import _read_and_merge

output_files = sorted(TW_STOCK_KBAR_1H.glob("*.parquet"))
rows = []
for f in tqdm(output_files, desc="checking"):
    d = f.stem
    src = _read_and_merge(d, TW_STOCK_KBAR_1MIN, TW_STOCK_KBAR_ABOVE_MA60)
    src_n = src["stock_id"].nunique() if not src.empty else 0
    out_n = pd.read_parquet(f, columns=["stock_id"])["stock_id"].nunique()
    rows.append({"date": d, "src_stocks": src_n, "out_stocks": out_n, "diff": src_n - out_n})

report = pd.DataFrame(rows)
mismatch = report[report["diff"] != 0]
print(f"總天數: {len(report)}, 不符天數: {len(mismatch)}")
display(mismatch.sort_values("diff", ascending=False).head(20))


checking: 100%|██████████| 1094/1094 [00:22<00:00, 49.52it/s]

總天數: 1094, 不符天數: 0


,date,src_stocks,out_stocks,diff


### backtest

#### config

In [6]:
import backtest

config = backtest.BacktestConfig(
    max_positions=10,  # 完整資金回測最多同時持有檔數
    max_position_pct=0.10,  # 單一標的完整倉位占初始資金比例
    first_entry_pct=0.50,  # 第一筆試單占單一完整倉位比例
    second_entry_pct=0.50,  # 第二筆加碼占單一完整倉位比例
    ma_fast=5,  # 快均線週期，單位為 1H K 根數
    ma_trend=55,  # 趨勢支撐均線週期，單位為 1H K 根數
    ma_mid=144,  # 中期均線週期，單位為 1H K 根數
    ma_long=200,  # 長期均線週期，單位為 1H K 根數
    vol_ma_window=20,  # 成交量均線週期，單位為 1H K 根數
    max_ma55_distance=0.25,  # 收盤價相對 55MA 的最高允許乖離
    support_tolerance=0.03,  # low 距支撐多近可算回踩
    pullback_volume_ratio=0.70,  # 回踩量縮門檻：volume < vol_ma * 此值
    breakout_volume_ratio=1.10,  # 確認突破量增門檻：volume > vol_ma * 此值
    breakout_setup_lookback=30,  # 55MA 突破後最多等待幾根 K 回踩
    min_runup_from_ma55=0.05,  # 突破後拉升至少高出突破時 55MA 的比例
    min_lift_volume_ratio=1.20,  # 拉升段最大量能至少是 vol_ma 的倍數
    max_lift_volume_ratio=3.00,  # 拉升段最大量能至多是 vol_ma 的倍數
    platform_lookback=12,  # 整理平台回看根數
    platform_max_range=0.08,  # 平台高低差占現價的最大比例
    platform_max_net_change=0.03,  # 平台起訖收盤淨變動的最大比例
    previous_low_lookback=12,  # 前低停損採用前幾根 K 的最低價
    defense_price_buffer=0.00,  # 防守價額外下方緩衝比例
    black_volume_ratio=2.50,  # 爆量長黑的量能門檻：volume / vol_ma
    black_body_pct=0.03,  # 爆量長黑的最小實體跌幅
    sector_return_window=20,  # 產業主流判斷的日報酬累積窗口
    sector_top_quantile=0.20,  # 近 20 日報酬排名前 20% 視為主流
    min_sector_members=5,  # 產業至少要有幾檔可交易成員才排名
    require_sector_main=True,  # entry_signal 是否必須屬於主流產業
    fee_rate=0.001425,  # 台股單邊牌告手續費率
    fee_discount=1,  # 實收手續費比例；1 表示不折扣
    sell_tax_rate=0.003,  # 賣出證交稅率
)

#### get signal df

In [ ]:
raise FileNotFoundError(
        f"{sector_map_file} is missing. Generate it from the note data update workflow."
    )


#### run backtest

In [18]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

import importlib
import backtest

importlib.reload(backtest)
from backtest import add_buy_and_hold_benchmark, run_signal_backtest

feature_df = pd.read_parquet("feature_df.parquet")
trades_df, equity_df = run_signal_backtest(
    feature_df,
    config=config,
)

benchmark_file = DATA_ROOT / "trading/tw_stock/kbar/1D/0050.parquet"
equity_df, benchmark_df = add_buy_and_hold_benchmark(
    equity_df,
    benchmark_price_file=benchmark_file,
    stock_id="0050",
    config=config,
)

equity_df[["datetime", "active_positions", "portfolio_return", "nav", "benchmark_nav"]].tail()

,datetime,active_positions,portfolio_return,nav,benchmark_nav
3738,2026-07-13 09:00:00,10,-0.021688,0.875133,3.472177
3739,2026-07-13 10:00:00,10,-0.000368,0.874811,3.472177
3740,2026-07-13 11:00:00,8,-0.003870,0.871425,3.472177
3741,2026-07-13 12:00:00,8,-0.001539,0.870084,3.472177
3742,2026-07-13 13:00:00,7,-0.002265,0.868113,3.472177


In [20]:
# Locate the Quant checkout before importing its shared configuration.
import os
import platform
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / "project_paths.py").is_file()),
    Path(os.environ.get(
        "QUANT_ROOT",
        r"C:\Users\user\Documents\GitHub\Quant"
        if platform.system() == "Windows"
        else "/Users/xinc/GitHub/Quant",
    )),
)
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
from project_paths import DATA_ROOT, NOTE_REPO_ROOT, QUANT_ROOT

import importlib
import backtest

importlib.reload(backtest)
from backtest import (
    add_buy_and_hold_benchmark,
    build_performance_metrics,
    plot_strategy_performance,
)

benchmark_file = DATA_ROOT / "trading/tw_stock/kbar/1D/0050.parquet"
equity_df, benchmark_df = add_buy_and_hold_benchmark(
    equity_df,
    benchmark_price_file=benchmark_file,
    stock_id="0050",
    config=config,
)


fig, performance_summary = plot_strategy_performance(equity_df)
fig.show()

performance_metrics = build_performance_metrics(equity_df, benchmark_df)
performance_metrics.style.format(
    {
        "Total Return": "{:.2%}",
        "CAGR": "{:.2%}",
        "Volatility": "{:.2%}",
        "Sharpe": "{:.2f}",
        "Max Drawdown": "{:.2%}",
        "Max DD Duration": "{:,.0f} days",
        "Profit Factor": "{:.2f}",
        "Win Rate": "{:.2%}",
        "Odds": "{:.2f}",
        "Avg Win": "{:.2%}",
        "Avg Loss": "{:.2%}",
        "Avg Return (Exp)": "{:.2%}",
        "Kelly": "{:.2f}",
    }
)

,Total Return,CAGR,Volatility,Sharpe,Max Drawdown,Max DD Duration,Profit Factor,Win Rate,Odds,Avg Win,Avg Loss,Avg Return (Exp),Kelly
Strategy,-13.19%,-3.28%,41.10%,0.11,-61.45%,"1,424 days",1.02,49.55%,1.04,2.13%,-2.05%,0.02%,0.01
Benchmark,247.22%,37.65%,23.62%,1.53,-28.47%,191 days,1.32,54.01%,1.13,1.12%,-1.00%,0.14%,0.13
